<a href="https://colab.research.google.com/github/chandanaka07-design/SSVDS/blob/main/SVVDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# prompt: generate a code to detect indian vehicles number plate in real time using dashcam or webcam

# Install necessary libraries
!pip install opencv-python easyocr

import cv2
import easyocr
import numpy as np

# Initialize EasyOCR reader
reader = easyocr.Reader(['en'], gpu=False) # Set gpu=True if you have a compatible GPU

def detect_number_plate(image_path):
    """
    Detects Indian vehicle number plates in an image.

    Args:
        image_path (str): Path to the image file.

    Returns:
        tuple: A tuple containing the processed image with bounding boxes
               and the detected number plate text. Returns (None, None) if
               no number plate is detected.
    """

    try:
        img = cv2.imread(image_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Use a more robust edge detection method
        edges = cv2.Canny(gray, 50, 150)

        # Find contours
        contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

        # Filter contours based on aspect ratio and area to find potential number plates
        number_plate_contour = None
        for contour in contours:
            perimeter = cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, 0.018 * perimeter, True)

            if len(approx) == 4:  # Check for quadrilateral shapes
                x, y, w, h = cv2.boundingRect(approx)
                aspect_ratio = w / float(h)
                if 2 < aspect_ratio < 5 and 1000 < w * h < 50000:  # Adjust these thresholds
                    number_plate_contour = approx
                    break

        if number_plate_contour is not None:
            x, y, w, h = cv2.boundingRect(number_plate_contour)
            number_plate_image = gray[y:y+h, x:x+w]

            # Apply OCR to the extracted number plate region
            results = reader.readtext(number_plate_image)

            text = ""
            for (bbox, text_, prob) in results:
              if prob > 0.5: # Adjust confidence threshold as needed
                  text += text_ + " "

            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(img, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

            return img, text.strip()  # Return the image and extracted text
        else:
            return None, None
    except Exception as e:
        print(f"Error processing image: {e}")
        return None, None



def process_video(video_path):
    """ Processes a video file and detects number plates in real-time. """

    cap = cv2.VideoCapture(video_path)  # Replace with your video file path or 0 for webcam
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Temporarily save the frame to a file (or use a memory buffer if possible)
        cv2.imwrite("temp_frame.jpg", frame)
        processed_frame, plate_text = detect_number_plate("temp_frame.jpg")

        if processed_frame is not None:
            cv2.imshow("Number Plate Detection", processed_frame)
            print(f"Detected Number Plate: {plate_text}")
        else:
            cv2.imshow("Number Plate Detection", frame)  # Display the original frame if no plate is found

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Example Usage (replace with your video or webcam input)
#process_video("your_video.mp4")
process_video(0) # For webcam




Error: Could not open video.


In [ ]:
# prompt: generate a code to detect the face actions in real time using a camera,for examples eyes are closed,facing away from the camera

import cv2
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

def detect_face_actions(video_path=0):
    """Detects face actions in real-time using a camera."""

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    with mp_face_mesh.FaceMesh(
        min_detection_confidence=0.5, min_tracking_confidence=0.5
    ) as face_mesh:
        while cap.isOpened():
            success, image = cap.read()
            if not success:
                print("Ignoring empty camera frame.")
                break

            # Flip the image horizontally for a later selfie-view display, and convert
            # the BGR image to RGB.
            image = cv2.cvtColor(cv2.flip(image, 1), cv2.COLOR_BGR2RGB)
            # To improve performance, optionally mark the image as not writeable to
            # pass by reference.
            image.flags.writeable = False
            results = face_mesh.process(image)

            # Draw the face mesh annotations on the image.
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.multi_face_landmarks:
                for face_landmarks in results.multi_face_landmarks:
                    # Get landmark positions
                    left_eye_landmarks = [face_landmarks.landmark[33], face_landmarks.landmark[160]]
                    right_eye_landmarks = [face_landmarks.landmark[263], face_landmarks.landmark[387]]

                    # Check for eyes closed
                    left_eye_distance = abs(left_eye_landmarks[0].y - left_eye_landmarks[1].y)
                    right_eye_distance = abs(right_eye_landmarks[0].y - right_eye_landmarks[1].y)
                    if left_eye_distance < 0.01 and right_eye_distance < 0.01 :
                      cv2.putText(image, 'Eyes Closed', (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1,(0,0,255),2)

                    #Example of detecting a face turning away (simplified)
                    #Get nose tip landmark
                    nose_tip = face_landmarks.landmark[1]
                    if nose_tip.x < 0.3 or nose_tip.x > 0.7:
                      cv2.putText(image, 'Facing Away', (50,100), cv2.FONT_HERSHEY_SIMPLEX, 1,(0,0,255),2)

                    mp_drawing.draw_landmarks(
                        image=image,
                        landmark_list=face_landmarks,
                        connections=mp_face_mesh.FACEMESH_TESSELATION,
                        landmark_drawing_spec=None,
                        connection_drawing_spec=mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1, circle_radius=1)
                    )

            cv2.imshow('MediaPipe Face Mesh', image)
            if cv2.waitKey(5) & 0xFF == 27:
                break
    cap.release()
    cv2.destroyAllWindows()

# Example usage:
detect_face_actions() # Use default webcam (0) or provide a video file path


ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
# prompt: generate a code to detect face using web cam in realtime

#This code is already equipped to detect faces using the webcam.
#Just run the detect_face_actions() function.  The existing code
#in the provided file already handles this.

# Example usage (already in the provided code):
detect_face_actions() # Use default webcam (0) or provide a video file path



NameError: name 'detect_face_actions' is not defined

In [ ]:
# prompt: generate a new code to detect face using webcam in realtime ,use webcam available in the pc

import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

# For webcam input:
cap = cv2.VideoCapture(0)
with mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.5) as face_detection:
  while cap.isOpened():
    success, image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      continue

    # To improve performance, optionally mark the image as not writeable to
    # pass by reference.
    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_detection.process(image)

    # Draw the face detection annotations on the image.
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    if results.detections:
      for detection in results.detections:
        mp_drawing.draw_detection(image, detection)
    # Flip the image horizontally for a selfie-view display.
    cv2.imshow('MediaPipe Face Detection', cv2.flip(image, 1))
    if cv2.waitKey(5) & 0xFF == 27:
      break
cap.release()
cv2.destroyAllWindows()


ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Load pre-trained models (e.g., YOLO for object detection and a custom model for helmet detection)
yolo_net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")
helmet_model = load_model("helmet_detection_model.h5")

# Load COCO labels for YOLO
with open("coco.names", "r") as f:
    classes = f.read().strip().split("\n")

# Lane detection parameters
lane_width = 3.7  # Standard lane width in meters
focal_length = 1.6  # Focal length of the camera in mm
pixel_to_meter = 0.002  # Conversion factor

# Function to detect vehicles and their speed
def detect_vehicles_and_speed(frame, yolo_net, classes):
    height, width, _ = frame.shape
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
    yolo_net.setInput(blob)
    layer_names = yolo_net.getLayerNames()
    output_layers = [layer_names[i[0] - 1] for i in yolo_net.getUnconnectedOutLayers()]
    detections = yolo_net.forward(output_layers)

    vehicles = []
    for output in detections:
        for detection in output:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if confidence > 0.5 and classes[class_id] in ["car", "truck", "motorcycle", "bus"]:
                box = detection[0:4] * np.array([width, height, width, height])
                (center_x, center_y, w, h) = box.astype("int")
                x = int(center_x - (w / 2))
                y = int(center_y - (h / 2))
                vehicles.append((classes[class_id], (x, y, w, h)))

    return vehicles

# Function to detect lane changes
def detect_lane_changes(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=50, minLineLength=100, maxLineGap=50)

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    return frame

# Function to detect rash driving (sudden lane changes or high speed)
def detect_rash_driving(vehicles, lane_changes, speed_threshold=80):
    rash_driving = False
    for vehicle in vehicles:
        speed = estimate_speed(vehicle)
        if speed > speed_threshold or lane_changes > 2:
            rash_driving = True
    return rash_driving

# Function to estimate speed of the vehicle
def estimate_speed(vehicle):
    # Placeholder for speed estimation logic
    # Use pixel displacement between frames and time difference
    return 60  # Example speed in km/h

# Function to detect helmets on two-wheelers
def detect_helmets(frame, helmet_model):
    resized_frame = cv2.resize(frame, (224, 224))
    normalized_frame = resized_frame / 255.0
    input_data = np.expand_dims(normalized_frame, axis=0)
    prediction = helmet_model.predict(input_data)
    return prediction[0][0] > 0.5  # Assuming binary classification (helmet/no helmet)

# Main loop for processing video
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect vehicles and their speed
        vehicles = detect_vehicles_and_speed(frame, yolo_net, classes)

        # Detect lane changes
        frame_with_lanes = detect_lane_changes(frame)

        # Detect rash driving
        rash_driving = detect_rash_driving(vehicles, lane_changes=0)

        # Detect helmets on two-wheelers
        for vehicle in vehicles:
            if vehicle[0] == "motorcycle":
                helmet_detected = detect_helmets(frame, helmet_model)
                if not helmet_detected:
                    cv2.putText(frame, "No Helmet!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # Display the output
        cv2.imshow("Dashcam Output", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Run the video processing
process_video("dashcam_video.mp4")

error: OpenCV(4.10.0) /io/opencv/modules/dnn/src/darknet/darknet_importer.cpp:210: error: (-212:Parsing error) Failed to open NetParameter file: yolov3.cfg in function 'readNetFromDarknet'


In [ ]:
!sudo apt install tesseract-ocr
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 18 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.1-2.1build1 [236 kB]
Fetched 4,816 kB in 2s (3,187 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debc

In [ ]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [ ]:
import cv2
import pytesseract
import easyocr
import numpy as np
from ultralytics import YOLO
import time

# Ensure Tesseract OCR path is set (Windows users might need to change this)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Load YOLO models
vehicle_model = YOLO('yolov8n.pt')  # Pretrained model for vehicles
obstacle_model = YOLO('yolov8n.pt')  # Pretrained model for obstacles
helmet_model = YOLO('yolov8n.pt')  # Pretrained model for helmet detection

# Load Indian RTO Codes (You need a dictionary mapping state codes to states)
rto_codes = {
    'KA': 'Karnataka', 'MH': 'Maharashtra', 'TN': 'Tamil Nadu',
    'DL': 'Delhi', 'GJ': 'Gujarat', 'UP': 'Uttar Pradesh'
}

# Initialize EasyOCR for Number Plate Recognition
reader = easyocr.Reader(['en'])

# Function to detect number plate and extract state
def detect_number_plate(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    plates = reader.readtext(gray)
    for result in plates:
        bbox, text, prob = result
        if prob > 0.5:
            state_code = text[:2].upper()
            state = rto_codes.get(state_code, 'Unknown')
            print(f"Detected Number: {text}, State: {state}")
            return text, state
    return None, None

# Function to detect vehicles and classify type
def detect_vehicles(frame):
    results = vehicle_model(frame)
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = vehicle_model.names[int(box.cls[0])]
            confidence = box.conf[0]
            if confidence > 0.5:
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return frame

# Function to detect lane changes
def detect_lane_changes(prev_frame, curr_frame):
    diff = cv2.absdiff(prev_frame, curr_frame)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
    return np.sum(thresh) > 50000  # Threshold for detecting significant movement

# Function to detect vehicle speed
def detect_speed(prev_position, curr_position, time_interval):
    distance = np.linalg.norm(np.array(curr_position) - np.array(prev_position))
    speed = distance / time_interval  # Simplified speed estimation
    return speed

# Function to detect rash driving
def detect_rash_driving(speed, lane_change_detected):
    if speed > 80 and lane_change_detected:
        return True
    return False

# Function to detect potholes
def detect_potholes(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        if cv2.contourArea(cnt) > 500:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
            cv2.putText(frame, "Pothole", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return frame

# Function to detect helmet usage
def detect_helmet(frame):
    results = helmet_model(frame)
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = helmet_model.names[int(box.cls[0])]
            if "helmet" not in label.lower():
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(frame, "No Helmet", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return frame

# Start Real-time Dashcam Feed
cap = cv2.VideoCapture(0)
prev_frame = None
prev_time = time.time()
prev_position = (0, 0)

if not cap.isOpened():
    print("Error: Could not open camera.")
else:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to capture frame.")
            break

        curr_time = time.time()
        if prev_frame is not None:
            lane_change_detected = detect_lane_changes(prev_frame, frame)
            curr_position = (cap.get(cv2.CAP_PROP_POS_FRAMES), cap.get(cv2.CAP_PROP_POS_AVI_RATIO))
            speed = detect_speed(prev_position, curr_position, curr_time - prev_time)
            rash_driving = detect_rash_driving(speed, lane_change_detected)
            if rash_driving:
                print("Warning: Rash Driving Detected!")
            prev_position = curr_position
            prev_time = curr_time

        # Apply all detection functions
        number, state = detect_number_plate(frame)
        frame = detect_vehicles(frame)
        frame = detect_potholes(frame)
        frame = detect_helmet(frame)

        prev_frame = frame.copy()
        cv2.imshow('Real-time Road Safety System', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


ModuleNotFoundError: No module named 'ultralytics'